# IMC 2025 — Adaptive Scene-Graph Matching

This notebook drives the coursework pipeline in `src/adaptive_imc/`.

**Idea:** use DINOv2 + ALIKED/LightGlue as public building blocks, but make the project contribution the **adaptive graph controller**: pair-quality estimation, spatial-coverage diagnosis, conditional tile/rotation rescue, triangle support, topology-aware rescue, and graph scene-label propagation.

The notebook intentionally does **not** reproduce the 1st-place MASt3R pipeline.

> A hidden Kaggle score cannot be guaranteed before execution. The workflow below treats **40 on the provided train metric as a gate**: validate `balanced`; if it falls short, run the more compute-heavy `high_score` preset before submitting.


## 0. One-time terminal setup

Run these in a terminal from the project root before using the notebook:

```bash
nix develop
bash scripts/bootstrap.sh
source .venv/bin/activate
python scripts/cache_models.py
bash scripts/download_data.sh
jupyter lab notebooks/AdaptiveSceneGraph_IMC2025.ipynb
```

If you opened Jupyter from `.venv`, the cells below use the correct environment automatically.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import pandas as pd
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

print("project:", ROOT)
print("python:", sys.executable)
print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
assert torch.cuda.is_available(), "Fix CUDA/Nix setup before running the image matcher."


## 1. Check the competition data

Expected files are `data/train/`, `data/test/`, `data/train_labels.csv`, `data/train_thresholds.csv`, and `data/sample_submission.csv`.


In [ ]:
DATA = ROOT / "data"
required = [
    DATA / "train",
    DATA / "train_labels.csv",
    DATA / "train_thresholds.csv",
]
for p in required:
    print("OK" if p.exists() else "MISSING", p)
assert all(p.exists() for p in required), "Run bash scripts/download_data.sh first."

print("train datasets:", sorted(p.name for p in (DATA/"train").iterdir() if p.is_dir()))


## 2. Smoke test

This is deliberately small/fast. It proves that DINO retrieval, ALIKED/LightGlue, geometric verification and COLMAP all function. **Do not judge the project score from `fast.yaml`.**


In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    subprocess.run([
        sys.executable, "scripts/run_pipeline.py",
        "--mode", "train",
        "--dataset", "ETs",
        "--config", "configs/fast.yaml",
        "--run-name", "smoke_ets",
    ], check=True)
    subprocess.run([
        sys.executable, "scripts/score_local.py",
        "outputs/smoke_ets/submission.csv",
    ], check=True)


## 3. Proposed method: balanced profile

This is the profile I would use for the assignment first. It keeps the **adaptive** character of the method instead of exhaustively applying tiling to every candidate pair.


In [ ]:
RUN_BALANCED_ETS = True

if RUN_BALANCED_ETS:
    subprocess.run([
        sys.executable, "scripts/run_pipeline.py",
        "--mode", "train",
        "--dataset", "ETs",
        "--config", "configs/balanced.yaml",
        "--run-name", "balanced_ets",
    ], check=True)
    subprocess.run([
        sys.executable, "scripts/score_local.py",
        "outputs/balanced_ets/submission.csv",
    ], check=True)


### Inspect what the router actually did

The diagnostics are useful for both debugging and the report: you can show how many pairs were accepted directly, rescued with tiles/rotation, or admitted because of triangle support / topology rescue.


In [ ]:
diag_path = ROOT / "outputs/balanced_ets/ETs/pair_diagnostics.csv"
if diag_path.exists():
    diag = pd.read_csv(diag_path)
    display(diag.sort_values(["accepted", "inliers"], ascending=[False, False]).head(30))
    print("routes:")
    display(diag.groupby(["route", "accepted"]).size().rename("pairs").reset_index())


In [ ]:
# Optional visualization: geometric quality of retrieved edges.
import matplotlib.pyplot as plt
if diag_path.exists():
    diag = pd.read_csv(diag_path)
    fig, ax = plt.subplots(figsize=(8,5))
    ax.scatter(diag["coverage"], diag["inliers"], alpha=.5)
    ax.set_xlabel("4x4 spatial coverage")
    ax.set_ylabel("MAGSAC inliers")
    ax.set_title("Pair geometry: coverage vs. inliers")
    plt.show()


## 4. Full train validation

Set `RUN_FULL_BALANCED=True` when the ETs run is healthy. This can take a long time depending on your GPU and the number of rescued pairs.

This is the decision-making score, not the smoke test.


In [ ]:
RUN_FULL_BALANCED = False  # flip to True when ready

if RUN_FULL_BALANCED:
    subprocess.run([
        sys.executable, "scripts/run_pipeline.py",
        "--mode", "train",
        "--config", "configs/balanced.yaml",
        "--run-name", "balanced_full",
    ], check=True)
    subprocess.run([
        sys.executable, "scripts/score_local.py",
        "outputs/balanced_full/submission.csv",
    ], check=True)


## 5. Score-oriented fallback

Only use this if the full `balanced` validation is below your desired margin. `high_score.yaml` keeps the same graph/triangle/topology system, but increases retrieval coverage and applies base+quadrant matching to many more pairs. It is intentionally expensive.


In [ ]:
RUN_HIGH_SCORE = False

if RUN_HIGH_SCORE:
    subprocess.run([
        sys.executable, "scripts/run_pipeline.py",
        "--mode", "train",
        "--config", "configs/high_score.yaml",
        "--run-name", "high_score_full",
    ], check=True)
    subprocess.run([
        sys.executable, "scripts/score_local.py",
        "outputs/high_score_full/submission.csv",
    ], check=True)


## 6. Generate the final test submission

Choose `FINAL_CONFIG` **after** comparing local validation. Do not silently change algorithms after seeing only a public leaderboard result; keep the train ablations for the report.


In [ ]:
RUN_TEST = False
FINAL_CONFIG = "configs/balanced.yaml"  # or configs/high_score.yaml

if RUN_TEST:
    subprocess.run([
        sys.executable, "scripts/run_pipeline.py",
        "--mode", "test",
        "--config", FINAL_CONFIG,
        "--run-name", "final_test",
    ], check=True)
    subprocess.run([
        sys.executable, "scripts/check_submission.py",
        "outputs/final_test/submission.csv",
        "--sample", "data/sample_submission.csv",
    ], check=True)
    print("Final file:", ROOT / "outputs/final_test/submission.csv")


## 7. Suggested ablation table for the report

Do not fill this with guessed values. Run the experiments and replace `NaN` with measured scores/runtime.


In [ ]:
ablation = pd.DataFrame([
    ["A", "DINO + ALIKED/LG baseline", np.nan, np.nan],
    ["B", "+ geometry/coverage edge filtering", np.nan, np.nan],
    ["C", "+ adaptive tile rescue", np.nan, np.nan],
    ["D", "+ rotation retry", np.nan, np.nan],
    ["E", "+ triangle + topology rescue (balanced)", np.nan, np.nan],
    ["F", "aggressive high-score profile", np.nan, np.nan],
], columns=["experiment", "method", "score", "runtime_min"])
ablation


## Interpretation / contribution statement

A clean way to present the work:

> We formulate image matching as adaptive computation over a scene graph. A cheap retrieval-and-local-matching pass estimates pair quality using geometric inliers and spatial support. Expensive crop/rotation views are invoked only for ambiguous pairs or images whose low graph degree risks splitting a true scene. Borderline edges can be rehabilitated by triangle support, and graph connectivity is also used to propagate scene membership to images that COLMAP does not register.

That accurately describes what this repository implements, while treating DINOv2, ALIKED, LightGlue, MAGSAC and COLMAP as cited building blocks rather than claiming them as original work.
